# Verify `kerr_schild_core.hpp` metric derivatives with sympy

Checks the analytic Kerr-Schild derivative formulas in `src/metrics/kerr_schild_core.hpp` (`compute_r_and_deriv`, `compute_l_and_deriv`, `compute_H_and_deriv`, `compute_inverse_metric_deriv`) against direct sympy differentiation of the closed-form `r(x,y,z)`, `l_mu(x,y,z)`, `H(x,y,z)`, `g^{mu nu}(x,y,z)`.

Each C++ formula is transcribed verbatim below. Instead of `sp.simplify()`-ing the `(C++ - true)` residual symbolically -- which hangs on these doubly-nested-sqrt expressions (`sqrt(A^2 + 4 a^2 z^2)` inside an outer `sqrt`) -- each residual is evaluated to 40 decimal digits at several random `(M, a, x, y, z)` points. Agreement to ~1e-32+ across many independent points is a standard way to confirm two closed forms are identically equal without waiting on symbolic simplification.

In [ ]:
import random
import sympy as sp

x, y, z, M, a = sp.symbols('x y z M a', real=True)

## `compute_r` / `compute_r_and_deriv`

In [ ]:
rsq = x**2 + y**2 + z**2
A = rsq - a**2
S = sp.sqrt(A**2 + 4*a**2*z**2)
r = sp.sqrt(sp.Rational(1, 2)*(A + S))

one_plus_A_over_S = 1 + A/S
dr_cpp = [
    x*one_plus_A_over_S/(2*r),                    # dr/dx
    y*one_plus_A_over_S/(2*r),                    # dr/dy
    (z*one_plus_A_over_S + 2*a**2*z/S)/(2*r),      # dr/dz
]
dr_true = [sp.diff(r, v) for v in (x, y, z)]

## `compute_l` / `compute_l_and_deriv` (chain rule through `r(x,y,z)`)

In [ ]:
B = r**2 + a**2
l = [sp.Integer(1), (r*x + a*y)/B, (r*y - a*x)/B, z/r]  # l0,l1,l2,l3 as explicit functions of x,y,z

vars_ = (x, y, z)
dl_true = [[sp.diff(l[mu], v) for mu in range(4)] for v in vars_]

dl_cpp = [[None]*4 for _ in range(3)]
for k, v in enumerate(vars_):
    dx_dk = 1 if v is x else 0
    dy_dk = 1 if v is y else 0
    dz_dk = 1 if v is z else 0
    dB_dk = 2*r*dr_cpp[k]
    dl_cpp[k][0] = sp.Integer(0)
    dl_cpp[k][1] = ((dr_cpp[k]*x + r*dx_dk) + a*dy_dk - l[1]*dB_dk)/B
    dl_cpp[k][2] = ((dr_cpp[k]*y + r*dy_dk) - a*dx_dk - l[2]*dB_dk)/B
    dl_cpp[k][3] = (dz_dk - l[3]*dr_cpp[k])/r

## `compute_H` / `compute_H_and_deriv` (chain rule through `r(x,y,z)`)

In [ ]:
D = r**4 + a**2*z**2
H = M*r**3/D

dH_true = [sp.diff(H, v) for v in vars_]
dH_cpp = []
for k, v in enumerate(vars_):
    dz_dk = 1 if v is z else 0
    dD_dk = 4*r**3*dr_cpp[k] + 2*a**2*z*dz_dk
    dH_cpp.append(M*(3*r**2*dr_cpp[k]*D - r**3*dD_dk)/D**2)

## `compute_metric` / `compute_inverse_metric`: Sherman-Morrison analytic inverse vs true matrix inverse

`g_{mu nu} = eta_{mu nu} + 2 H l_mu l_nu` and, since `l` is null under `eta`, the exact inverse is `g^{mu nu} = eta^{mu nu} - 2 H l^mu l^nu` (Sherman-Morrison).

In [ ]:
eta = sp.diag(-1, 1, 1, 1)
eta_inv = eta  # eta is its own inverse
l_col = sp.Matrix(l)
g_cov = eta + 2*H*l_col*l_col.T

lup = eta_inv*l_col  # raise index with eta: lup = (-l0, l1, l2, l3)
g_con_analytic = eta_inv - 2*H*lup*lup.T

## `compute_inverse_metric_deriv`: `dginv[alpha][mu][nu] = d(g^{mu nu})/dX^alpha`

In [ ]:
dginv_true = [[[sp.diff(g_con_analytic[mu, nu], v) for nu in range(4)] for mu in range(4)] for v in vars_]
dginv_cpp = [[[None]*4 for _ in range(4)] for _ in range(3)]
for k, v in enumerate(vars_):
    dlup_k = [-dl_cpp[k][0], dl_cpp[k][1], dl_cpp[k][2], dl_cpp[k][3]]
    for mu in range(4):
        for nu in range(4):
            dginv_cpp[k][mu][nu] = -2*(dH_cpp[k]*lup[mu]*lup[nu] + H*(dlup_k[mu]*lup[nu] + lup[mu]*dlup_k[nu]))

## Numeric equality checks

Evaluate both sides at several random `(M, a, x, y, z)` points to 40 decimal digits.

In [ ]:
PREC = 40
N_TRIALS = 6
random.seed(0)


def random_point():
    a_val = sp.nsimplify(round(random.uniform(0.2, 0.98), 6))
    M_val = sp.nsimplify(round(random.uniform(0.5, 2.0), 6))
    # keep rho^2 = x^2+y^2+z^2 comfortably outside the horizon region
    x_val = sp.nsimplify(round(random.uniform(-6, 6), 6))
    y_val = sp.nsimplify(round(random.uniform(-6, 6), 6))
    z_val = sp.nsimplify(round(random.uniform(-6, 6), 6))
    return {a: a_val, M: M_val, x: x_val, y: y_val, z: z_val}


def max_abs_residual(expr_cpp, expr_true):
    worst = sp.Float(0)
    for _ in range(N_TRIALS):
        pt = random_point()
        val_cpp = expr_cpp.subs(pt).evalf(PREC)
        val_true = expr_true.subs(pt).evalf(PREC)
        worst = max(worst, abs(val_cpp - val_true))
    return worst


TOL = sp.Float(10) ** (-(PREC - 8))  # generous slack below full precision
print(f"Numeric check: {N_TRIALS} random (M,a,x,y,z) points, {PREC}-digit precision, tol={TOL}")

In [ ]:
print("=== compute_r_and_deriv ===")
for k, v in enumerate(vars_):
    resid = max_abs_residual(dr_cpp[k], dr_true[k])
    print(f"  dr/d{v}: max|residual| = {resid}  ({'OK' if resid < TOL else 'MISMATCH'})")

In [ ]:
print("=== compute_l_and_deriv ===")
for k, v in enumerate(vars_):
    for mu in range(4):
        resid = max_abs_residual(dl_cpp[k][mu], dl_true[k][mu])
        print(f"  d(l_{mu})/d{v}: max|residual| = {resid}  ({'OK' if resid < TOL else 'MISMATCH'})")

In [ ]:
print("=== compute_H_and_deriv ===")
for k, v in enumerate(vars_):
    resid = max_abs_residual(dH_cpp[k], dH_true[k])
    print(f"  dH/d{v}: max|residual| = {resid}  ({'OK' if resid < TOL else 'MISMATCH'})")

In [ ]:
print("=== Sherman-Morrison inverse identity (g_cov^-1 == g_con_analytic) ===")
worst = sp.Float(0)
for _ in range(N_TRIALS):
    pt = random_point()
    g_cov_num = g_cov.subs(pt).evalf(PREC)
    g_con_num = g_con_analytic.subs(pt).evalf(PREC)
    g_cov_inv_num = g_cov_num.inv()  # cheap: purely numeric 4x4 inverse
    diff_mat = g_cov_inv_num - g_con_num
    worst = max(worst, max(abs(e) for e in diff_mat))
print(f"  max|residual entry| = {worst}  ({'OK' if worst < TOL else 'MISMATCH'})")

In [ ]:
print("=== compute_inverse_metric_deriv ===")
mismatches = 0
checked = 0
for k, v in enumerate(vars_):
    for mu in range(4):
        for nu in range(4):
            checked += 1
            resid = max_abs_residual(dginv_cpp[k][mu][nu], dginv_true[k][mu][nu])
            if resid >= TOL:
                mismatches += 1
                print(f"  d(g^{{{mu}{nu}}})/d{v}: max|residual| = {resid}  (MISMATCH)")

print(f"done: {mismatches} mismatch(es) out of {checked} (alpha,mu,nu) components checked")